<a href="https://colab.research.google.com/github/IlyaShlapak2003/-HomeWork7/blob/master/%D0%92%D1%82%D0%BE%D1%80%D0%BE%D0%B9_%D0%BA%D0%B5%D0%B9%D1%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2 кейс

**Соберите данные из Google Sheets и напишите функцию `generate_report`, которая принимает три списка (данные с каждого листа в Google Sheets) и сохраняет отчет в файл `student_debt_report.txt`**

**Важно**

Перед началом решения задачи выполните следующую ячейку - в ней скачиваются нужные файлы

In [1]:
!wget https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json

!wget https://gist.github.com/Vs8th/39c5deed0f5539d781f00328f7fd4fe0/raw/result.txt

--2026-11-01 19:41:45--  https://gist.github.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.github.com (gist.github.com)... 140.82.116.3
Connecting to gist.github.com (gist.github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json [following]
--2026-11-01 19:41:46--  https://gist.githubusercontent.com/Vs8th/d0bd4bdbbb58c8ae4f70a2a503e2d5fc/raw/creds.json
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2358 (2.3K) [text/plain]
Saving to: ‘creds.json’

creds.json          100%[===================>]   2.30K  --.-KB/s    in 0s      

2026-11-01 19:41:46 (43.9 MB/s) - ‘creds.json’ saved [2

Чтобы посмотреть как выглядят сами гугл таблицы в оригинале - можете перейти по [ссылке](https://docs.google.com/spreadsheets/d/1hRnw-PEftF0J-6KY7InlILVwWdkJu4vJiGwRjuE_P4U/edit?usp=sharing) и изучить.

Небольшое описание столбцов в таблицах:  
**Лист1:** `student_id` - айди студента; `student_name` - имя студента; `installment` (Y/N) - факт наличия рассрочки у студента (Y - рассрочка есть, N - рассрочки нет).  
**Лист2:** `student_id` - айди студента; `last_payment_date` - дата последней полученной оплаты; `expected_payment_date` - дата, в которую ожидаем следующий платеж (рассчитывается от даты последнего платежа).  
**Лист3:** `student_id` - айди студента; `already_payed_amount` - уже выплаченная часть рассрочки; `left_to_pay` - оставшаяся (невыплаченная) часть; `one-time_payment` - единоразовый платеж; `installment_amount` - общая сумма, которая бралась в рассрочку.

Как примерно должен выглядеть результат:

In [2]:
with open('result.txt') as f:
  print(f.read())

Студент Иванов У.У. - долг 100000 рублей
Студент Петров Е.Е. - долг 250000 рублей
Студент Сидоров И.И. - долг 10000 рублей


In [7]:
#@title Если возникнут трудности с определением `scope`, подсказка:

scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
         'https://www.googleapis.com/auth/drive']

**Примечание**

Считать должников необходимо на 1 марта 2023 года. То есть определяя количество просроченных платежей, мы определяем их количество не на настоящую дату, а на **1 марта 2023 года**. А периоды внесения платежей для всех студентов одинаковы - **6 месяцев** (183 дня).

То есть, если ожидаемый платеж должен был пройти 3 февраля 2022 года, то к 1 марта 2023 студент просрочил уже три платежа (3 февраля 2022, 3 августа 2022 и 3 февраля 2023). При расчетах отталкивайтесь от даты ожидаемого платежа, разницу между платежами берите 183 дня.

**Решение**

Напишите свое решение ниже

In [28]:

from datetime import datetime
def generate_report(sheet1, sheet2, sheet3):
    report_date = datetime(2023, 3, 1)
    payment_period = 183
    def parse_date(value):
        if isinstance(value, datetime):
            return value
        value = str(value).strip()
        for fmt in ("%Y-%m-%d", "%d.%m.%Y", "%d/%m/%Y",
                    "%Y-%m-%d %H:%M:%S"):
            try:
                return datetime.strptime(value, fmt)
            except ValueError:
                continue
        raise ValueError(f"Неверный формат даты: {value}")

    def to_number(value):
        return float(str(value).replace(" ", "").replace(",", "."))
    def normalize_id(value):
        return str(value).strip().removesuffix(".0")
    dates = {
        normalize_id(row["student_id"]): row
        for row in sheet2
    }
    payments = {
        normalize_id(row["student_id"]): row
        for row in sheet3
    }
    result = []
    for student in sheet1:
        if str(student["installment"]).strip().upper() != "Y":
            continue
        student_id = normalize_id(student["student_id"])
        if student_id not in dates or student_id not in payments:
            continue
        payment_date = parse_date(
            dates[student_id]["expected_payment_date"]
        )
        if payment_date > report_date:
            continue
        days_overdue = (report_date - payment_date).days
        missed_payments = days_overdue // payment_period + 1
        payment_info = payments[student_id]
        one_time_payment = to_number(
            payment_info["one-time_payment"]
        )
        installment_amount = to_number(
            payment_info["installment_amount"]
        )
        already_paid = to_number(
            payment_info["already_payed_amount"]
        )
        remaining_debt = max(
            0, installment_amount - already_paid
        )
        debt = min(
            missed_payments * one_time_payment,
            remaining_debt
        )
        if debt > 0:
            name = student["student_name"]
            result.append(
                f"Студент {name} - долг {debt:g} рублей"
            )
    with open("student_debt_report.txt", "w", encoding="utf-8") as file:
        file.write("\n".join(result))

In [30]:

import gspread
from oauth2client.service_account import ServiceAccountCredentials
scope = [
    "https://www.googleapis.com/auth/spreadsheets.readonly",
    "https://www.googleapis.com/auth/drive.readonly"
]
creds = ServiceAccountCredentials.from_json_keyfile_name(
    "creds.json", scope
)
client = gspread.authorize(creds)
spreadsheet = client.open("Installments")
sheet1 = spreadsheet.worksheet("Лист1").get_all_records()
sheet2 = spreadsheet.worksheet("Лист2").get_all_records()
sheet3 = spreadsheet.worksheet("Лист3").get_all_records()
generate_report(sheet1, sheet2, sheet3)

In [31]:

import os
print(os.path.exists("student_debt_report.txt"))
with open("student_debt_report.txt", encoding="utf-8") as f:
    print(f.read()[:1000])

True
Студент Кузнецова К.А. - долг 266666 рублей
Студент Петров К.А. - долг 66666 рублей
Студент Иванов А.А. - долг 100000 рублей
Студент Иванов П.П. - долг 116666 рублей
Студент Петров И.И. - долг 116666 рублей
Студент Петров А.А. - долг 16666 рублей
Студент Иванов Е.В. - долг 41666 рублей
Студент Николаева Е.В. - долг 100000 рублей
Студент Смирнова Е.В. - долг 233333 рублей
Студент Николаева М.А. - долг 33332 рублей
Студент Кузнецова А.А. - долг 133332 рублей
Студент Иванов А.А. - долг 25000 рублей
Студент Петров И.И. - долг 133332 рублей
Студент Кузнецова К.А. - долг 133333 рублей
Студент Кузнецова Е.В. - долг 66666 рублей
Студент Сидоров И.И. - долг 83333 рублей
Студент Петров И.И. - долг 16666 рублей
Студент Кузнецова И.И. - долг 233333 рублей
Студент Иванов А.А. - долг 266666 рублей
Студент Сидоров К.А. - долг 266666 рублей
Студент Смирнова И.И. - долг 200000 рублей
Студент Сидоров М.А. - долг 166666 рублей
Студент Сидоров Е.В. - долг 33333 рублей
Студент Иванов Е.В. - долг 10000

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [32]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt

import pandas as pd

user_answer = pd.read_csv('student_debt_report.txt')
correct_answer = pd.read_csv('student_debt.txt')

--2026-11-01 20:17:47--  https://gist.github.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.github.com (gist.github.com)... 140.82.116.3
Connecting to gist.github.com (gist.github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt [following]
--2026-11-01 20:17:47--  https://gist.githubusercontent.com/Vs8th/63832f093f4db8d8b251ba5d39571f3d/raw/student_debt.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11325 (11K) [text/plain]
Saving to: ‘student_debt.txt.4’

student_debt.txt.4  100%[===================>]  11.06K  --.-KB/s    in 0s      

2026-11-01 20:17:49 (97.6 MB/

In [33]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
  assert user_answer.columns.equals(correct_answer.columns), 'Названия столбцов не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!
